# Physical roofs and conditioned basements

This offline lesson builds one declared example, inspects separate ceiling, roof and ground elements, and preserves a cooler conditioned basement. No weather, network or EnergyPlus is required. The named preset, 18 °C basement and 0.5 attic air changes per hour are illustrative assumptions, not evidence for a real building.

In [1]:
from epsm_data.domain.enums import BuildingElement as Role

from epsm_core import Building, Footprint, GeometrySpec, RoofSpec, ScheduleSpec
from epsm_core.specs import UnconditionedExchangeSpec, ZoneUseSpec

reference = Building(preset="illustrative-reference")
geometry = GeometrySpec.from_footprint(
    Footprint.rectangle(10, 8), storeys=2, storey_height_m=3,
    basement_depth_m=2.5, basement_conditioned=True,
    roof=RoofSpec(shape="gable", pitch_deg=30, direction_deg=90),
)
basement_use = reference.operation.with_changes(
    heating_setpoint=ScheduleSpec.constant(18),
    occupant_density_per_m2=0, equipment_gain_w_m2=0, dhw_kwh_m2_year=0,
)
building = Building(
    preset="illustrative-reference", geometry=geometry,
    zone_uses=(ZoneUseSpec(zone_kind="basement", use=basement_use,
                           source="declared 18 C illustrative basement"),),
    unconditioned_exchange=(UnconditionedExchangeSpec(
        zone_kind="attic", outdoor_air_ach=0.5,
        source="declared illustrative attic outdoor exchange"),),
)
print(building)
print(building.readiness())

Building(identifier='building', parts=1, elements=27, stages=0, preset='illustrative-reference')
Readiness(ready=True, missing=0, unsupported=0)


The occupied ceiling (`TOP_FLOOR`) and sloped weather shell (`ROOF`) are separate installations. The floor at grade lies between conditioned main storeys and basement; it is internal. Selecting both an insulated ceiling and insulated shell represents both constructions and needs justification in a real study.

In [2]:
for element in building.elements:
    if element.role in {Role.ROOF, Role.TOP_FLOOR, Role.GROUND_FLOOR, Role.BASEMENT_FLOOR}:
        print(element.identifier, element.role.value, round(element.area_m2, 3), element.boundary)
assert next(e for e in building.elements if e.identifier == "main.top").boundary == "Internal"
assert next(e for e in building.elements if e.identifier == "main.ground_floor").boundary == "Internal"
assert next(e for e in building.elements if e.identifier == "main.basement_floor").boundary == "Ground"
assert sum(e.area_m2 for e in building.elements if e.role == Role.ROOF) > 80
assert building.geometry.conditioned_floor_area_m2 == 240
assert building.readiness("quantities").ready
print("Material occurrences:", len(building.quantities()))

main.basement_floor GROUND_FLOOR 80.0 Ground
main.ground_floor GROUND_FLOOR 80.0 Internal
main.top TOP_FLOOR 80.0 Internal
main.roof ROOF 92.376 Outdoors
Material occurrences: 47


A draft without constructions and systems remains incomplete. Geometry does not infer its roof recipe or ventilation. A snapshot preserves explicit zone controls and exchange declarations; changing a later scenario does not rewrite it. RC additionally checks whether its selected reduction supports the complete controls and geometry.

In [3]:
draft = Building(geometry=geometry, facade=reference.facade)
assert "construction:main.roof" in draft.readiness().missing
print(draft.readiness())

snapshot = building.snapshot()
restored = snapshot.to_building()
assert restored.zone_uses == building.zone_uses
assert restored.unconditioned_exchange == building.unconditioned_exchange
assert restored.snapshot().sha256 == snapshot.sha256
print("Controls and physical inventory survive the snapshot roundtrip.")

Readiness(ready=False, missing=32, unsupported=0)
Controls and physical inventory survive the snapshot roundtrip.


Touching pitched parts, occupied roof rooms, partial basements and unconditioned-basement RC reduction remain unsupported. See [physical roof and basement inputs](../docs/roofs-and-basements.md) for ownership, migration and boundary assumptions. Accepted geometry is not evidence that the model matches EPC heating.